# Stream preprocessing (galaxy → cells → GMM chunks)

Uses **`cpen.apps.streams.preprocess`** (no `improved_interaction_networks` for cutflow / plots).

Composition: **DES background + mock streams** (real/S5 members dropped). Mock streams with **$<100$ stars** are demoted to field at load — those objects are too tiny to supervise. Cut thresholds are chosen with **`is_mock_stream`** on this whole mock galaxy, then applied to unseen mock galaxies.

Checkpoint dir: `/scratch/gpfs/BHANIN/jgdezoort/streams`.

Pipeline: drop tiny mock streams → DES 5° cells → physical CMD + $\pi<1$ mas → **GMM $k=64$ in proper motion** (partition, not a membership cut) → drop $w>W$ → **one graph per remaining Gaussian blob**, including empty field blobs.


In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

# 0. Preprocessing the Stream Dataframes

The input DES+mock galaxy files are located in

`/projects/BHANIN/adri_gage_gnn_streams/gnn_streams/mocks/galaxy_with_background/`

Focus galaxy: `train/train_galaxy_0000_with_background.parquet`.

Built by `scans/streams/0_preprocess_stream_df.py`:

- Load + photometry cuts; build `stream_label`, **`is_mock_stream`**, etc.
- Drop real/S5 stream members → **DES background + mocks only**.
- **Demote mock streams with $<100$ stars to field** (galaxy-wide count). Stars stay in the table; they are no longer labelled as stream members.
- Assign DES footprint circles; write checkpoint + per-cell tangent-plane parquets.

**Cut studies below use `is_mock_stream` (kept mock members in this galaxy), not `is_train`.**
Thresholds chosen here will later be applied to *unseen* mock galaxies.

If you load an older checkpoint that still contains real streams, the load cells filter them out. Tiny-stream demotion is also applied on load so cached GMM labels pick up the new flags.


In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from cpen.apps.streams.preprocess.config import MIN_MOCK_STREAM_STARS
from cpen.apps.streams.preprocess.des_footprint_centers import DES_FOOTPRINT_CENTERS
from cpen.apps.streams.preprocess.galaxy import (
    drop_real_stream_members,
    drop_tiny_mock_streams,
    ensure_is_mock_stream,
    tiny_mock_stream_labels,
)
from cpen.apps.streams.preprocess.plotting import plot_sky_with_circles
from cpen.apps.streams.preprocess.utils import print_mock_stream_cell_occupancy


CKPT_DIR = Path("/scratch/gpfs/BHANIN/jgdezoort/streams")
df = pd.read_parquet(CKPT_DIR / "0000_checkpoint.parquet")
df = drop_real_stream_members(df, verbose=True)
df = ensure_is_mock_stream(df)
TINY_MOCK_STREAMS = tiny_mock_stream_labels(df, min_stars=MIN_MOCK_STREAM_STARS)
df = drop_tiny_mock_streams(df, tiny_labels=TINY_MOCK_STREAMS, verbose=True)
meta = yaml.safe_load((CKPT_DIR / "0000_meta.yaml").read_text()) if (CKPT_DIR / "0000_meta.yaml").exists() else {}
circle_radius = meta.get("circle_radius_deg", 5.0)


df_out, centers = plot_sky_with_circles(
    df,
    centers=DES_FOOTPRINT_CENTERS,
    circle_radius=circle_radius,
    max_points=300_000,
    use_precomputed_cells=True,
    color_by_assignment=True,
    verbose=False,
)

# Per mock stream: star count, #cells occupied, and cell fractions (overlap → Σ can exceed 1)
mock_cell_occ = print_mock_stream_cell_occupancy(df)


In [ ]:
from pathlib import Path

import pandas as pd

from cpen.apps.streams.preprocess.config import MIN_MOCK_STREAM_STARS
from cpen.apps.streams.preprocess.galaxy import (
    drop_real_stream_members,
    ensure_is_mock_stream,
    prepare_mock_cells,
    tiny_mock_stream_labels_from_frames,
)
from cpen.apps.streams.preprocess.plotting import plot_cell_three_panel

# On-disk cell parquets may still contain real/S5 streams from an older checkpoint.
cell_dfs = {}
for f in Path("/scratch/gpfs/BHANIN/jgdezoort/streams/cells").glob("cell_*.parquet"):
    cid = int(f.stem.split("_")[1])
    df_c = drop_real_stream_members(pd.read_parquet(f), verbose=False)
    cell_dfs[cid] = ensure_is_mock_stream(df_c)

# Galaxy-wide tiny-stream labels (not per-cell occupancy).
if "TINY_MOCK_STREAMS" not in globals():
    TINY_MOCK_STREAMS = tiny_mock_stream_labels_from_frames(
        cell_dfs.values(), min_stars=MIN_MOCK_STREAM_STARS
    )
cell_dfs = prepare_mock_cells(
    cell_dfs, min_stream_stars=MIN_MOCK_STREAM_STARS, tiny_labels=TINY_MOCK_STREAMS
)

plot_cell_three_panel(cell_dfs, [1,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30,32,64])


# 1. Simple Cutflow

## 1.0 Simple Demonstration

Streams appear localized in a corner of a parallax vs. g−r diagram.
Below we compare **mock-stream** members to everything else (DES background + non-members),
first in a few example cells and then aggregated over the union of every DES cell
(overlapping stars counted once).


In [ ]:
from cpen.apps.streams.preprocess.des_footprint_centers import DES_FOOTPRINT_CENTERS
from cpen.apps.streams.preprocess.galaxy import prepare_mock_table
from cpen.apps.streams.preprocess.plotting import plot_aggregate_parallax_cmd

df = pd.read_parquet("/scratch/gpfs/BHANIN/jgdezoort/streams/0000_checkpoint.parquet")
df = prepare_mock_table(df, tiny_labels=globals().get("TINY_MOCK_STREAMS"), verbose=True)
centers = DES_FOOTPRINT_CENTERS

for j in [2, 27, 100, 93, 73]:
    col = f"in_cell_{j + 1}"
    if col not in df.columns:
        continue
    cell_id = j + 1
    sub = df[df[col]].copy()
    is_mock = sub["is_mock_stream"].to_numpy(bool)

    g = sub["psf_mag_aper_8_g_corrected_des"].to_numpy()
    r = sub["psf_mag_aper_8_r_corrected_des"].to_numpy()
    plx = sub["parallax_gaia"].to_numpy()
    g_r = g - r

    mg = np.isfinite(g) & (g < 100) & (g > -100)
    mr = np.isfinite(r) & (r < 100) & (r > -100)
    mplx = np.isfinite(plx) & (plx > -100) & (plx < 100)
    mask = mg & mr & mplx

    print(
        f"Cell {cell_id}: N={len(sub):,}  mock={is_mock.sum():,}  "
        f"other={(~is_mock).sum():,}"
    )

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.scatter(
        g_r[~is_mock & mask], plx[~is_mock & mask],
        s=2, c="steelblue", alpha=0.4, label="other",
    )
    ax.scatter(
        g_r[is_mock & mask], plx[is_mock & mask],
        s=15, c="crimson", alpha=0.8, marker="^", label="mock stream",
    )
    ax.set_xlabel("g - r (mag)")
    ax.set_ylabel("parallax (mas)")
    ax.set_title(f"Cell {cell_id}: parallax vs g-r")
    ax.legend()
    plt.tight_layout()
    plt.show()

# Union of every DES cell (overlaps counted once): hexbin of other + all mocks
plot_aggregate_parallax_cmd(df)


## 1.1 CMD Cuts

Mock streams appear localized at or below a certain threshold in g−r.
The per-cell CMDs below are followed by the same all-cell aggregate as in §1.0,
now in $(g-r,\,r)$ (bright at top). A dashed line marks the mock 90th percentile
of $g-r$ as a reference one-sided color cut.


In [ ]:
from cpen.apps.streams.preprocess.galaxy import prepare_mock_table
from cpen.apps.streams.preprocess.plotting import plot_aggregate_cmd, plot_cell_cmd

plot_cell_cmd(cell_dfs, [2, 27, 73])

# Union of every DES cell (overlaps counted once). Uses `df` from §0 if present.
if "df" not in globals():
    df = pd.read_parquet("/scratch/gpfs/BHANIN/jgdezoort/streams/0000_checkpoint.parquet")
    df = prepare_mock_table(df, tiny_labels=globals().get("TINY_MOCK_STREAMS"), verbose=True)

plot_aggregate_cmd(df)


In [ ]:
from cpen.apps.streams.preprocess.galaxy import prepare_mock_table

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sample from the original dataframe
df = pd.read_parquet("/scratch/gpfs/BHANIN/jgdezoort/streams/0000_checkpoint.parquet")
df = prepare_mock_table(df, tiny_labels=globals().get("TINY_MOCK_STREAMS"), verbose=True)

g = df["psf_mag_aper_8_g_corrected_des"].to_numpy()
r = df["psf_mag_aper_8_r_corrected_des"].to_numpy()
g_r = g - r

parallax = df["parallax_gaia"].to_numpy()

is_mock = df["is_mock_stream"].to_numpy(bool)

mask = np.isfinite(g_r) & np.isfinite(parallax) & (g_r >= -100) & (g_r <= 100) & (r >= -100) & (r <= 100)
g_r = g_r[mask]
parallax = parallax[mask]
is_mock = is_mock[mask]

n_mock_total = int(is_mock.sum())
n_other_total = int((~is_mock).sum())

g_r_thresholds = np.linspace(0, 2, 30)
rows = []
for thresh in g_r_thresholds:
    keep = g_r <= thresh
    n_mock = int((is_mock & keep).sum())
    n_other = int((~is_mock & keep).sum())
    rows.append({
        "threshold": thresh,
        "n_mock": n_mock,
        "n_other": n_other,
        "mock_frac": n_mock / n_mock_total if n_mock_total > 0 else np.nan,
        "other_frac": n_other / n_other_total if n_other_total > 0 else np.nan,
        "purity": n_mock / (n_mock + n_other),
    })
df_sweep = pd.DataFrame(rows)

print("===== Points with mock_frac>0.9 and other_frac<0.6 ======")
display(df_sweep[(df_sweep.mock_frac>0.9) & (df_sweep.other_frac<0.6)])

# choose a specific threshold to plot
chosen = df_sweep.iloc[11]

# plot sig v background and purity v threshold 
fig, axs = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

# plot 1: mock vs other kept when cutting g-r > threshold
ax = axs[0]
ax.plot(df_sweep.other_frac, df_sweep.mock_frac, "o-", markersize=4)
ax.set_xlabel("Other kept")
ax.set_ylabel("Mock kept")
ax.set_title("Keep g-r ≤ threshold")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
#ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

# plot the chosen threshold
ax.axvline(chosen["other_frac"], label=f"Threshold={chosen['threshold']:.2f}", color="red", linestyle="--")
ax.legend(loc="best")

# plot 2: purity vs. g-r threshold 
ax = axs[1]
ax.plot(df_sweep.threshold, df_sweep.purity, "o-", markersize=4)
ax.set_xlabel("(g-r) threshold")
ax.set_ylabel("Purity")
ax.set_title("Purity vs. Threshold (g-r ≤ threshold)")
#ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

# plot the chosen threshold 
ax.axvline(chosen["threshold"], label=f"Mock Eff={chosen['mock_frac']:.2f}", color="red", linestyle="--")
ax.legend(loc="best")

plt.tight_layout()
plt.show()


A vertical $g-r$ cut is a blunt rectangle. The stream locus is closer to a **linear isochrone** $r = m(g-r)+b$. The next cell fits that line to the mocks (PCA), then keeps a **parallel slab** (two diagonals with the same $m$, different intercepts) and sweeps the half-width — and, separately, the slope.


In [ ]:
from cpen.apps.streams.preprocess.plotting import (
    cmd_slab_intercepts,
    fit_linear_cmd_isochrone,
    plot_aggregate_cmd,
    sweep_cmd_slab,
)

g = df["psf_mag_aper_8_g_corrected_des"].to_numpy()
r_mag = df["psf_mag_aper_8_r_corrected_des"].to_numpy()
g_r = g - r_mag
is_mock = df["is_mock_stream"].to_numpy(bool)

g_r_lim, r_lim = (-0.6, 2.4), (15.0, 24.0)
m_win = (
    np.isfinite(g_r) & np.isfinite(r_mag)
    & (g_r >= g_r_lim[0]) & (g_r <= g_r_lim[1])
    & (r_mag >= r_lim[0]) & (r_mag <= r_lim[1])
)
g_r_w, r_w, mock_w = g_r[m_win], r_mag[m_win], is_mock[m_win]

iso = fit_linear_cmd_isochrone(g_r_w, r_w, mock_w)
m, b = iso["m"], iso["b"]
print(f"PCA isochrone:  r = {m:.3f} (g-r) + {b:.3f}")

# Width that keeps 90% of mocks; overlay that slab on the aggregate CMD
half_widths = np.linspace(0.02, 1.2, 40)
df_slab = sweep_cmd_slab(g_r_w, r_w, mock_w, m, b, half_widths)
hit90 = df_slab[df_slab.mock_frac >= 0.9]
chosen_w = float(hit90.iloc[0]["half_width"]) if len(hit90) else float(df_slab.iloc[-1]["half_width"])
b_lo, b_hi = cmd_slab_intercepts(m, b, chosen_w)
pick90 = hit90.iloc[0] if len(hit90) else df_slab.iloc[-1]
print(
    f"Half-width at mock_frac≥0.9: w={chosen_w:.3f} mag  "
    f"(other_frac={pick90.other_frac:.3f}, purity={pick90.purity:.4f})"
)

plot_aggregate_cmd(
    df, mark_mock_quantile=None, slab=(m, b_lo, b_hi),
)

# Compare slab vs vertical g-r cut
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
ax = axs[0]
ax.plot(df_slab.other_frac, df_slab.mock_frac, "o-", markersize=4, label="linear slab")
# reuse vertical sweep if present, else compute
if "df_sweep" in globals() and "threshold" in df_sweep.columns:
    ax.plot(df_sweep.other_frac, df_sweep.mock_frac, "s--", markersize=4, label="vertical g−r")
ax.set_xlabel("Other kept")
ax.set_ylabel("Mock kept")
ax.set_title("CMD cut: slab vs vertical")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend()

ax = axs[1]
ax.plot(df_slab.half_width, df_slab.purity, "o-", markersize=4)
ax.axvline(chosen_w, color="red", ls="--", label=f"w={chosen_w:.2f} (90% mock)")
ax.set_xlabel("slab half-width (mag)")
ax.set_ylabel("Purity")
ax.set_title("Purity vs. slab width")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# Slope scan: for each m, center b on mocks, take w at 90% mock efficiency
slopes = np.linspace(-20.0, 2.0, 23)  # steep (MS-like) → shallow (RGB-like)
rows = []
ws = np.linspace(0.02, 1.5, 35)
for m_try in slopes:
    b_try = float(np.median(r_w[mock_w] - m_try * g_r_w[mock_w]))
    sw = sweep_cmd_slab(g_r_w, r_w, mock_w, m_try, b_try, ws)
    ok = sw[sw.mock_frac >= 0.9]
    if len(ok) == 0:
        pick = sw.iloc[-1]
    else:
        pick = ok.iloc[0]
    rows.append({
        "m": m_try,
        "b": b_try,
        "half_width": pick.half_width,
        "mock_frac": pick.mock_frac,
        "other_frac": pick.other_frac,
        "purity": pick.purity,
    })
df_slope = pd.DataFrame(rows)
best = df_slope.loc[df_slope.other_frac.idxmin()]
print("===== Slope scan (w chosen for mock_frac≥0.9; lower other_frac is better) =====")
display(df_slope.sort_values("other_frac").head(8))
print(
    f"Best slope in scan: m={best.m:.3f}, b={best.b:.3f}, w={best.half_width:.3f}  "
    f"other_frac={best.other_frac:.3f}, purity={best.purity:.4f}"
)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(df_slope.m, df_slope.other_frac, "o-", markersize=4)
ax.axvline(m, color="0.4", ls=":", label=f"PCA m={m:.2f}")
ax.axvline(best.m, color="red", ls="--", label=f"best m={best.m:.2f}")
ax.set_xlabel("slope m  in  r = m(g−r)+b")
ax.set_ylabel("other kept (at 90% mock)")
ax.set_title("Linear isochrone: which slope rejects the most background?")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 1.2 Parallax as a nearby-disk veto

The stars are the **DES sample**. DES does not measure parallax; `parallax_gaia`
is the Gaia astrometric measurement on the DES×Gaia crossmatch.

Halo streams are distant, so a **one-sided upper bound** $\pi < \pi_{\max}$
can reject nearby disk stars. There is **no lower bound**: a faint DES star's
Gaia parallax scatters through $\pi \sim 0$, including negatives.

Do **not** tune $\pi_{\max}$ on these mocks (their $\pi$ is unsmeared truth).
Choose a conservative threshold that a noisy parallax ($\sigma_\pi \sim 0.5$–$1$ mas)
would still pass. Default: $\pi < 1$ mas.


In [ ]:
from cpen.apps.streams.preprocess.plotting import plot_parallax_upper_bound_sweep

# Conservative foreground cut: keep π < 1 mas (no lower bound).
# Tune against realistic parallax errors on DES stars, not the unsmeared mock spike.
PI_MAX = 1.0  # mas
plot_parallax_upper_bound_sweep(df, chosen=PI_MAX)


## 2. GMM chunks in proper motion

Each DES cell is still $\sim 10^4$–$10^5$ stars. The next split has to be kinematic: a cold stream is localized in $(\mu_\phi,\mu_\lambda)$, while a sky or CMD sub-bin is arbitrary.

**GMM, not DBSCAN.** A mixture of $k$ Gaussians is a complete tessellation of the cell — every star with finite PM is assigned, $k$ is fixed, and a cold stream is approximately Gaussian after convolution with measurement error. DBSCAN (and HDBSCAN) would be noisier here: $\varepsilon$ / `min_samples` are extra knobs, the number of clusters jumps from cell to cell, and the leftover noise label ($-1$) is an awkward extra bin. The DES field is a continuous disk+halo density, not islands in a void.

**$k=64$ is the working default** (tried 16 then 32: stream share stayed ~99%, but host field blobs were still $\sim 6$–$11$k stars). Treat $k$ as a graph-size knob, not something to tune on `mock_frac`. Stage 3 then drops Gaussians with $w>W=1.65\,\mathrm{mas\,yr}^{-1}$ (~99% unique-star mock keep at this $k$). Each remaining `(cell, Gaussian)` is one graph.

**Keep every component at fit time.** Do not pick the blob with the most mock stars. Empty cells give negative-control chunks; that is the point. The width cut in §2.2 is a later, label-free filter on Gaussian size.

**Localization QC** (after the fit): for each cell with enough mocks, did one Gaussian collect them, or are they smeared across $k$? Metrics are *share* (completeness in the richest blob), *enrichment* (purity of that blob / cell mock fraction; $1\times$ = random), and mock vs field PM RMS inside the host blob. Tags (`localized` / `split` / `dilute`) are a fit check, not a cut.

Covariance floor: $\sigma\geq 0.5\,\mathrm{mas\,yr}^{-1}$ (`PM_FLOOR_MAS_YR`), a Gaia-like error, so unsmeared mock PMs cannot collapse onto a delta-function component (that would be label leak in the partition itself).


In [ ]:
from cpen.apps.streams.preprocess.config import MIN_MOCK_STREAM_STARS
from cpen.apps.streams.preprocess.galaxy import (
    drop_real_stream_members,
    ensure_is_mock_stream,
    prepare_mock_cells,
    tiny_mock_stream_labels_from_frames,
)
from cpen.apps.streams.preprocess.gmm import (
    GMM_N_COMPONENTS,
    PM_FLOOR_MAS_YR,
    assign_gmm_to_cells,
    gmm_component_table,
    load_gmm_cells,
    occupancy_from_cells,
    save_gmm_cells,
)
from cpen.apps.streams.preprocess.plotting import plot_cell_gmm, plot_gmm_occupancy

CKPT_DIR = Path("/scratch/gpfs/BHANIN/jgdezoort/streams")
GMM_K = GMM_N_COMPONENTS  # 64
# n_init=1 / 20k subsample: interactive speed. Bump for the production graph build.
GMM_FIT_KW = dict(n_init=1, max_fit_samples=20_000, pm_floor=PM_FLOOR_MAS_YR, random_state=0)

if "TINY_MOCK_STREAMS" not in globals():
    TINY_MOCK_STREAMS = None

if "cell_dfs" not in globals() or not cell_dfs:
    cell_dfs = {}
    for f in (CKPT_DIR / "cells").glob("cell_*.parquet"):
        cid = int(f.stem.split("_")[1])
        cell_dfs[cid] = ensure_is_mock_stream(
            drop_real_stream_members(pd.read_parquet(f), verbose=False)
        )

cached = load_gmm_cells(CKPT_DIR, subdir=f"cells_gmm_k{GMM_K}")
gmm_fits = {}
if cached is not None and all("gmm_label" in df.columns for df in cached.values()):
    cell_dfs = cached
    print(f"Loaded cached k={GMM_K} GMM labels for {len(cell_dfs)} cells")
    print(f"Ellipses omitted (cache has labels only). Re-run without cells_gmm_k{GMM_K} to refit.")
else:
    print(f"Fitting GMM k={GMM_K} (pm_floor={PM_FLOOR_MAS_YR} mas/yr) on {len(cell_dfs)} cells…")
    cell_dfs, gmm_fits, gmm_occ = assign_gmm_to_cells(
        cell_dfs, n_components=GMM_K, verbose=True, **GMM_FIT_KW
    )
    out = save_gmm_cells(cell_dfs, CKPT_DIR, subdir=f"cells_gmm_k{GMM_K}")
    print(f"Wrote {out}")

if TINY_MOCK_STREAMS is None:
    TINY_MOCK_STREAMS = tiny_mock_stream_labels_from_frames(
        cell_dfs.values(), min_stars=MIN_MOCK_STREAM_STARS
    )
cell_dfs = prepare_mock_cells(
    cell_dfs, min_stream_stars=MIN_MOCK_STREAM_STARS, tiny_labels=TINY_MOCK_STREAMS
)
gmm_occ = occupancy_from_cells(cell_dfs, n_components=GMM_K)

from cpen.apps.streams.preprocess.gmm import print_gmm_occupancy
print_gmm_occupancy(gmm_occ, n_components=GMM_K)
display(gmm_occ.sort_values("n_mock", ascending=False).head(12))


In [ ]:
from cpen.apps.streams.preprocess.gmm import (
    mock_stream_localization,
    occupancy_from_cells,
    print_gmm_localization,
)
from cpen.apps.streams.preprocess.plotting import (
    plot_cell_gmm,
    plot_gmm_localization,
    plot_gmm_mock_bars,
    plot_gmm_occupancy,
)

# Rebuild occupancy so QC columns exist even if the fit ran before this code was added.
gmm_occ = occupancy_from_cells(cell_dfs, n_components=GMM_K)
print_gmm_localization(gmm_occ, n_components=GMM_K)

stream_loc, stream_chunks = mock_stream_localization(cell_dfs)
print()
print("===== Per mock stream (can be localized in a cell and still split across DES cells) =====")
display(stream_loc.head(20))
if len(stream_loc):
    print(
        f"  median share in one (cell, GMM) chunk: {100 * stream_loc.best_share.median():.0f}%   "
        f"m|edian n_chunks={stream_loc.n_chunks.median():.0f}  "
        f"median n_cells={stream_loc.n_cells.median():.0f}"
    )

plot_gmm_occupancy(gmm_occ, min_mock=30)
plot_gmm_localization(gmm_occ, min_mock=30)
plot_gmm_mock_bars(cell_dfs, gmm_occ, n_show=6, n_components=GMM_K)

# Per cell: PM tessellation + mocks, then sky|PM for each mock-hosting Gaussian
SHOW_CELLS = []
if "qc_tag" in gmm_occ.columns:
    for tag in ("localized", "split", "dilute"):
        sub = gmm_occ[(gmm_occ.qc_tag == tag) & (gmm_occ.n_mock >= 30)]
        if len(sub):
            SHOW_CELLS.append(int(sub.sort_values("n_mock", ascending=False).iloc[0].cell_id))
for cid in (2, 27, 100, 93, 73):
    if cid in cell_dfs and cid not in SHOW_CELLS:
        SHOW_CELLS.append(cid)
SHOW_CELLS = [c for c in SHOW_CELLS if c in cell_dfs][:5]
plot_cell_gmm(cell_dfs, gmm_fits, SHOW_CELLS)

if len(gmm_occ) and gmm_occ.n_mock.max() > 0:
    rich_id = int(gmm_occ.loc[gmm_occ.n_mock.idxmax(), "cell_id"])
    print(f"===== Components in cell {rich_id} (most mocks; keep all {GMM_K}) =====")
    display(
        gmm_component_table(cell_dfs[rich_id], cell_id=rich_id)
        .sort_values("n_mock", ascending=False)
        .reset_index(drop=True)
    )


## 2.2 GMM width cut (sky-wide)

Reject Gaussians that are too wide: keep stars with $w_{\mathrm{GMM}} \le W$, where $w$ is the largest RMS axis of the assigned component (mas/yr).

Aggregation is **unique stars** over the DES footprint (overlapping cells counted once). A star that sits in several cells is assigned the **minimum** width among those GMMs — it survives the cut if *any* cell puts it in a Gaussian narrower than $W$. That is the sky-wide completeness of applying the same per-cell cut on another galaxy.

Do **not** hug the mock-optimal $W$. Mock proper motions are unsmeared, so mock efficiency is optimistic relative to Gaia-smeared streams. Take the $W$ at high mock keep (e.g. 90–95%) and prefer rounding **up**. The covariance floor is already $0.5\,\mathrm{mas\,yr}^{-1}$.

In [ ]:
from cpen.apps.streams.preprocess.gmm import GMM_WIDTH_MAX, sweep_gmm_width
from cpen.apps.streams.preprocess.plotting import plot_gmm_width_sweep

gmm_width_sweep, gmm_width_unique, gmm_width_ops = sweep_gmm_width(cell_dfs)
display(gmm_width_ops)

# Working cut for Stage 3 (k=64): ~99% unique-star mock-keep knee. Sweep above is the justification.
print(f"Stage 3 GMM_WIDTH_MAX = {GMM_WIDTH_MAX:.3f} mas/yr  (k={GMM_K} ~99% unique-star mock-keep knee)")

is_mock_u = gmm_width_unique["is_mock_stream"].to_numpy(bool)
w_u = gmm_width_unique["gmm_width"].to_numpy(float)
plot_gmm_width_sweep(
    gmm_width_sweep,
    chosen=GMM_WIDTH_MAX,
    width_mock=w_u[is_mock_u],
    width_other=w_u[~is_mock_u],
)


## 2.3 Previous working cut: $k=32,\,W=2$ (optional)

$k=64,\,W=1.65$ is the default (§2). This cell loads `cells_gmm_k32/` only for a matched-completeness comparison.

$W$ has to move with $k$. Comparing $k=64$ at the $k=32$ cut ($W=2$) is the wrong operating point: extra components make field Gaussians narrower, so they flood under a cut that was tuned for $k=32$. At matched $\sim 99\%$ mock keep, $k=64$ uses $W\approx 1.65$ and yields smaller chunks (median remaining Gaussian $\sim 2.8$k vs $\sim 6.3$k).


In [ ]:
from cpen.apps.streams.preprocess.gmm import (
    apply_width_cut,
    gmm_component_table,
    load_gmm_cells,
    occupancy_from_cells,
    print_gmm_localization,
    print_gmm_occupancy,
    print_width_cut_summary,
    summarize_width_cut,
    sweep_gmm_width,
)
from cpen.apps.streams.preprocess.plotting import plot_gmm_components

cell_dfs_k32 = load_gmm_cells(CKPT_DIR, subdir="cells_gmm_k32")
if cell_dfs_k32 is None:
    print("cells_gmm_k32 not cached; skip k=32 comparison (k=64 is the default).")
else:
    occ32 = occupancy_from_cells(cell_dfs_k32, n_components=32)
    print_gmm_occupancy(occ32, n_components=32)
    print_gmm_localization(occ32, n_components=32)

    W32 = 2.0
    sw32, _, _ = sweep_gmm_width(cell_dfs_k32)
    print("\nk=32 unique-star keep at selected W (nearest sweep rows):")
    for w in (1.65, W32, 2.09):
        row = sw32.iloc[(sw32.threshold - w).abs().argmin()]
        print(f"  W≈{w:.2f} (sweep {row.threshold:.3f}): mock_keep={100*row.mock_frac:.1f}%  other_keep={100*row.other_frac:.1f}%")

    print(f"\n===== k=32 Stage 3 at W={W32} (compare to k=64 at W=1.65) =====")
    tbl32 = summarize_width_cut(cell_dfs_k32, width_max=W32, min_mock=100)
    print_width_cut_summary(tbl32, min_mock=100, width_max=W32)

    print(f"\n===== Cell 103 remaining Gaussians after w≤{W32} (k=32) =====")
    t = gmm_component_table(cell_dfs_k32[103], cell_id=103)
    t = t[t.width <= W32].sort_values("n_stars", ascending=False)
    print(t.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    cut103 = apply_width_cut({103: cell_dfs_k32[103]}, width_max=W32, min_mock=None)
    plot_gmm_components(cut103, [103], min_mock=1, max_per_cell=6)


## 3. Width cut + remaining blobs

**Label-free (carries to other galaxies):** drop every Gaussian with $w > 1.65\,\mathrm{mas\,yr}^{-1}$ (`GMM_WIDTH_MAX` at $k=64$). That is a kinematic cut on the GMM, not a DES-cell drop.

**Keep every remaining blob.** Tiny mock *streams* ($<100$ stars) were already demoted at load. After CMD / GMM / $w\le W$, empty field Gaussians stay in the graph pool as negatives. Occupancy below is a diagnostic of where the kept streams sit, not a keep/drop.


In [ ]:
from cpen.apps.streams.preprocess.gmm import (
    GMM_WIDTH_MAX,
    MIN_TRAIN_MOCK,
    apply_width_cut,
    gmm_component_table,
    print_width_cut_summary,
    summarize_width_cut,
)
from cpen.apps.streams.preprocess.plotting import plot_gmm_components, plot_train_mock_fractions

W = float(globals().get("GMM_WIDTH_MAX", GMM_WIDTH_MAX))

stage3_tbl = summarize_width_cut(cell_dfs, width_max=W, min_mock=int(MIN_TRAIN_MOCK))
cut_tbl = print_width_cut_summary(stage3_tbl, min_mock=int(MIN_TRAIN_MOCK), width_max=W)
display(stage3_tbl.sort_values(["n_mock_after", "n_after"], ascending=False).head(16))

# Occupancy diagnostic only (not a training gate).
plot_train_mock_fractions(stage3_tbl, min_mock=int(MIN_TRAIN_MOCK), width_max=W)

# Remaining stars in *every* cell after dropping w>W
cut_cell_dfs = apply_width_cut(cell_dfs, width_max=W)
print(f"Cells after w≤{W}: {len(cut_cell_dfs)}")

# Examples: mock-richest, an empty cell, a mid occupancy cell, a sparse leftover
show = []
if len(stage3_tbl):
    show.append(int(stage3_tbl.sort_values("n_mock", ascending=False).iloc[0].cell_id))
    empty = stage3_tbl[stage3_tbl.n_mock <= 0]
    if len(empty):
        show.append(int(empty.sort_values("n_after", ascending=False).iloc[0].cell_id))
    mid = stage3_tbl[stage3_tbl.n_mock > 0].sort_values("n_mock")
    if len(mid):
        show.append(int(mid.iloc[len(mid)//2].cell_id))
        show.append(int(mid.iloc[0].cell_id))
seen = set()
SHOW_CELLS = []
for c in show:
    if c in cut_cell_dfs and c not in seen:
        seen.add(c)
        SHOW_CELLS.append(c)

# One figure per remaining Gaussian. min_mock=0 includes empty field blobs.
plot_gmm_components(
    cut_cell_dfs,
    SHOW_CELLS,
    fits=gmm_fits if "gmm_fits" in globals() else {},
    min_mock=0,
    max_per_cell=6,
)

for cid in SHOW_CELLS:
    row = stage3_tbl[stage3_tbl.cell_id == cid].iloc[0]
    print(
        f"Cell {cid}: mock {int(row.n_mock_after)}/{int(row.n_mock)} kept  "
        f"N={int(row.n_after):,}  mock_frac {100*row.mock_frac:.3f}% → {100*row.mock_frac_after:.3f}%  "
        f"({int(row.n_comp_kept)} Gaussians left)"
    )
    display(
        gmm_component_table(cut_cell_dfs[cid], cell_id=cid)
        .sort_values("n_mock", ascending=False)
        .reset_index(drop=True)
    )


## 3.2 Graphs for CPEN: sky kNN + sky/PM hyperedges

Each remaining `(cell, Gaussian)` after $w\le W$ is one graph — **including empty field blobs**. Blobs with $N>8000$ stars are then **sky-split** (k-means; GMM optional) so no graph is a 50k-star field clump. Construction is **label-free**; mock flags are only for truth labels.

**kNN 2-edges** ($k=8$, sky): local on-sky neighbourhood. `edge_y = 1` iff both endpoints are mocks.

**Sky hyperedges** (cap 64, $k_{\mathrm{ball}}=16$, contrast $1.1$): on-sky overdensities **and** underdensities. Overlay: **blue circle** = overdensity, **red circle** = underdensity.

**PM hyperedges** (same peak knobs): overdensity peaks only in $(\mu_\phi,\mu_\lambda)$. Overlay: **blue square**. The GMM already grouped the blob; these pick colder patches *inside* that Gaussian.

`hyper_y = 1` iff member mock fraction $\ge 0.5$. Do not feed `kind` / peak-vs-under as `edge_type`. Soft edge labels come later.

The plots below sample mock-rich, mid, sparse, and empty blobs in **sky and proper motion**.


In [ ]:
from cpen.apps.streams.preprocess.blob_graph import (
    BLOB_HYPER_BG_MIN,
    BLOB_HYPER_CONTRAST,
    BLOB_HYPER_K,
    BLOB_HYPER_N,
    BLOB_HYPER_PURITY,
    BLOB_KNN_K,
    build_blob_hypers,
    build_blob_knns,
    pick_blob_examples,
    print_blob_hyper_summary,
    print_blob_knn_summary,
    print_blob_label_summary,
)
from cpen.apps.streams.preprocess.gmm import (
    BLOB_N_MAX,
    GMM_WIDTH_MAX,
    print_blob_summary,
    split_gmm_blobs,
    split_oversized_blobs,
)
from cpen.apps.streams.preprocess.plotting import plot_blob_examples

W = float(globals().get("GMM_WIDTH_MAX", GMM_WIDTH_MAX))

blobs = split_gmm_blobs(cell_dfs, width_max=W)
blobs = split_oversized_blobs(blobs, n_max=BLOB_N_MAX, method="kmeans")
blob_tab = print_blob_summary(blobs)

print(f"\nBuilding sky kNN (k={BLOB_KNN_K}) on {len(blobs):,} blobs…")
blob_knns = build_blob_knns(blobs, k=BLOB_KNN_K)
knn_tab = print_blob_knn_summary(blob_knns, blobs)

print(
    f"\nBuilding peak+background hyperedges  "
    f"(peak cap={BLOB_HYPER_N}, k_ball={BLOB_HYPER_K}, contrast={BLOB_HYPER_CONTRAST:g}, "
    f"bg_min={BLOB_HYPER_BG_MIN}, τ={BLOB_HYPER_PURITY:g})…"
)
blob_hypers = build_blob_hypers(
    blobs,
    n_hyper=BLOB_HYPER_N,
    k_ball=BLOB_HYPER_K,
    contrast=BLOB_HYPER_CONTRAST,
    bg_min=BLOB_HYPER_BG_MIN,
)
hyper_tab = print_blob_hyper_summary(blob_hypers, blobs)

# Keep every blob. Occupancy is a label, not a filter.
label_tab = print_blob_label_summary(blob_knns, blob_hypers, blobs)

example_tab = pick_blob_examples(
    blob_tab,
    n_rich=3,
    n_mid=2,
    n_sparse=2,
    n_empty=4,
    max_n_stars=25_000,
)
plot_blob_examples(blobs, blob_knns, blob_hypers, example_tab)


## 3.3 Which blobs to train on

**Keep all remaining blobs after $w\le W$.** Tiny mock streams ($<100$ stars) were dropped at load; from there the sky is partitioned and every CMD / GMM blob is a graph, even if it has no stream in it.

That is the point of the empty blobs: they are the negative class. Sparse leftover mocks in a field Gaussian are still labelled (`edge_y` / `hyper_y` / node $y$), just rare — they are not a reason to throw the pointing away.

Co-training later is still $L_{\mathrm{node}} + \lambda_2 L_{\mathrm{kNN}} + \lambda_H L_{\mathrm{hyper}}$ on this full pool. Occupancy tables above are for QC, not a second cut.


## 3.1 Mock-track diagnostics (optional)

PCA tubes around mock centroids are **not** the training volume. Stage 3 graphs are GMM components with $w\le 1.65$ in every DES cell (empty field blobs included).


In [ ]:
from cpen.apps.streams.preprocess.tracks import fit_all_mock_tracks, mock_track_summary
from cpen.apps.streams.preprocess.plotting import (
    plot_mock_track_detail,
    plot_mock_track_grid,
    plot_track_width_sweep,
)

if "df" not in globals():
    df = pd.read_parquet("/scratch/gpfs/BHANIN/jgdezoort/streams/0000_checkpoint.parquet")
    df = drop_real_stream_members(df, verbose=True)
    df = ensure_is_mock_stream(df)

MIN_TRACK_STARS = 30
TUBE_W = 0.35  # deg; diagnostic only

tracks = fit_all_mock_tracks(df, min_stars=MIN_TRACK_STARS)
track_tbl = mock_track_summary(tracks)
print(f"Fitted {len(tracks)} mock tracks (N ≥ {MIN_TRACK_STARS}) — diagnostic, not a window")
display(track_tbl)
plot_mock_track_grid(df, tracks, half_width=TUBE_W)

by_n = max(tracks.values(), key=lambda t: t.n_mock)
by_l = max(tracks.values(), key=lambda t: t.s_max - t.s_min)
print(f"Richest: {by_n.stream_label}  N={by_n.n_mock}  L={by_n.s_max-by_n.s_min:.1f}°  σ⊥={by_n.rms_perp:.3f}°")
print(f"Longest: {by_l.stream_label}  N={by_l.n_mock}  L={by_l.s_max-by_l.s_min:.1f}°  σ⊥={by_l.rms_perp:.3f}°")

plot_mock_track_detail(df, by_n, half_width=TUBE_W)
if by_l.stream_label != by_n.stream_label:
    plot_mock_track_detail(df, by_l, half_width=TUBE_W)

plot_track_width_sweep(df, tracks, chosen=TUBE_W)


# 4. Physical cutflow (then GMM chunks)

Parallax $\pi<1$ mas is the nearby-disk veto from §1.2. The $g-r$ number below is still the provisional vertical cut from §1.1; swap in the linear isochrone slab when that is frozen.

GMM width $W$ (`GMM_WIDTH_MAX` from §2.2) drops wide field Gaussians. After the physical cuts, report how many GMM chunks remain and how mocks sit in them.


In [ ]:
PI_MAX = 1.0  # mas; §1.2
G_R_MAX = 0.76  # provisional vertical CMD; replace with slab when frozen
from cpen.apps.streams.preprocess.gmm import GMM_WIDTH_MAX as _W_LIB
if "GMM_WIDTH_MAX" not in globals():
    GMM_WIDTH_MAX = _W_LIB  # k=64 default from gmm.py
    print(f"GMM_WIDTH_MAX not set; using {GMM_WIDTH_MAX} mas/yr")

rows = []
for name, cell_df in cell_dfs.items():
    sub = ensure_is_mock_stream(cell_df)
    g = sub["psf_mag_aper_8_g_corrected_des"].to_numpy()
    r = sub["psf_mag_aper_8_r_corrected_des"].to_numpy()
    g_r = g - r
    plx = sub["parallax_gaia"].to_numpy()
    is_mock = sub["is_mock_stream"].to_numpy(bool)

    phys = (
        np.isfinite(g_r) & np.isfinite(r)
        & (g_r >= -100) & (g_r <= 100) & (r >= -100) & (r <= 100)
        & (g_r <= G_R_MAX)
        & ((~np.isfinite(plx)) | (plx < PI_MAX))
    )
    if "gmm_width" in sub.columns and "gmm_label" in sub.columns:
        lab = sub["gmm_label"].to_numpy(int)
        w = sub["gmm_width"].to_numpy(float)
        phys = phys & (lab >= 0) & (w <= GMM_WIDTH_MAX)
    n_mock = int(is_mock.sum())
    n_other = int((~is_mock).sum())
    n_mock_post = int((is_mock & phys).sum())
    n_other_post = int((~is_mock & phys).sum())

    n_comp = np.nan
    n_comp_mock = np.nan
    if "gmm_label" in sub.columns:
        lab = sub["gmm_label"].to_numpy(int)
        keep = phys & (lab >= 0)
        n_comp = int(np.unique(lab[keep]).size) if keep.any() else 0
        mock_keep = keep & is_mock
        n_comp_mock = int(np.unique(lab[mock_keep]).size) if mock_keep.any() else 0

    if n_mock >= 30:
        print(
            f"Cell {name}: mock {n_mock_post}/{n_mock} ({100 * n_mock_post / n_mock:.1f}%)  "
            f"other {n_other_post}/{n_other} ({100 * n_other_post / max(n_other, 1):.1f}%)  "
            f"GMM chunks used {n_comp}  (mocks in {n_comp_mock})"
        )
    rows.append({
        "cell_id": name,
        "n_mock_before": n_mock,
        "n_mock_after": n_mock_post,
        "n_other_before": n_other,
        "n_other_after": n_other_post,
        "n_gmm_chunks": n_comp,
        "n_gmm_chunks_with_mock": n_comp_mock,
    })

cut_tbl = pd.DataFrame(rows)
print()
print("===== Cells with ≥30 mocks =====")
display(cut_tbl[cut_tbl.n_mock_before >= 30].sort_values("n_mock_before", ascending=False).head(15))


# 5. Nested PM GMM (optional)

$k=64$ on the cell is Stage 2; $w\le W$ plus a sky k-means/GMM split of blobs with $N>8000$ is the default graph unit (§3.2). This section is only for experiments that further tessellate in proper motion. Same rule: partition, do not cut on width, do not select the mock-rich sub-blob.


In [ ]:
from cpen.apps.streams.preprocess.galaxy import ensure_is_mock_stream

def save_gmm_subcells(cell_dfs, base_dir, subdir="cells_gmm2", width_thresh=None, verify=True):
    outdir = Path(base_dir) / subdir
    outdir.mkdir(parents=True, exist_ok=True)

    stats_rows = []
    n_files = 0
    saved_paths = []

    for cell_id, df in cell_dfs.items():
        if "gmm_label" not in df.columns or "gmm_label_2" not in df.columns:
            continue
        sub = df.copy()
        if width_thresh is not None:
            sub = sub[sub["gmm_width_2"] <= width_thresh]

        for (k, j), part in sub.groupby(["gmm_label", "gmm_label_2"]):
            if k < 0 or j < 0:
                continue
            if len(part) == 0:
                continue

            k_int, j_int = int(k), int(j)
            n_stars = len(part)
            is_mock = part["is_mock_stream"].to_numpy(bool)
            n_mock = int(is_mock.sum())
            n_other = int((~is_mock).sum())
            purity = n_mock / (n_mock + n_other) if (n_mock + n_other) > 0 else np.nan

            fp = outdir / f"cell_{cell_id:04d}_k{k_int:02d}_j{j_int:02d}.parquet"
            part.to_parquet(fp, index=False)
            n_files += 1
            saved_paths.append(fp)
            stats_rows.append({
                "cell_id": cell_id,
                "k": k_int,
                "j": j_int,
                "n_stars": n_stars,
                "n_mock": n_mock,
                "n_other": n_other,
                "purity": purity,
            })

    stats_df = pd.DataFrame(stats_rows)

    print(f"Saved {n_files} subcell files to {outdir}")
    if width_thresh is not None:
        print(f"Width cut: gmm_width_2 <= {width_thresh:.3f}")
    print()
    print("===== Subcell statistics =====")
    display(stats_df)
    print()
    print("===== Aggregate =====")
    print(f"  Total stars:       {stats_df['n_stars'].sum():,}")
    print(f"  Total mock:       {stats_df['n_mock'].sum():,}")
    print(f"  Total other:   {stats_df['n_other'].sum():,}")
    if stats_df["n_mock"].sum() + stats_df["n_other"].sum() > 0:
        agg_purity = stats_df["n_mock"].sum() / (stats_df["n_mock"].sum() + stats_df["n_other"].sum())
        print(f"  Purity:            {agg_purity:.4f}")
    print(f"  Subcells:          {len(stats_df)}")

    if verify and saved_paths:
        print()
        print("===== Verification =====")
        for fp in saved_paths[:min(5, len(saved_paths))]:
            df_check = pd.read_parquet(fp)
            n_comp = df_check["gmm_label_2"].nunique()
            k_j = (df_check["gmm_label"].iloc[0], df_check["gmm_label_2"].iloc[0])
            print(f"  {fp.name}: {len(df_check)} rows, (k,j)={k_j} -> {'OK' if n_comp == 1 else 'FAIL'}")

    return outdir

def load_gmm_subcells(base_dir, subdir="cells_gmm2"):
    outdir = Path(base_dir) / subdir
    if not outdir.exists() or not outdir.is_dir():
        return None
    cell_dfs = {}
    for fp in sorted(outdir.glob("cell_*.parquet")):
        split = fp.stem.split("_")
        cell_id, j, k = split[1], split[2][1:], split[3][1:]
        print(f"loaded {cell_id}, {j}, {k}")
        cell_dfs[(cell_id, j, k)] = ensure_is_mock_stream(pd.read_parquet(fp))
    return cell_dfs if cell_dfs else None


In [ ]:
from cpen.apps.streams.preprocess.gmm import fit_pm_gmm

GMM_N_COMPONENTS_2 = 8

subcell_dfs = load_gmm_subcells(CKPT_DIR, subdir="cells_gmm2")

if subcell_dfs is None:
    subcell_dfs = {}
    for name, cell_df in cell_dfs.items():
        print(name)
        cell_id = name
        sub = cell_df.copy()

        pm_phi = sub["pm_phi"].to_numpy()
        pm_lam = sub["pm_lam"].to_numpy()

        if "gmm_label" not in sub.columns:
            continue
        labels_1 = sub["gmm_label"].to_numpy()
        n_comp_1 = int(labels_1.max()) + 1 if len(labels_1) else 0

        gmm_label_2 = np.full(len(sub), -1)
        gmm_width_2 = np.full(len(sub), np.nan)

        for k in range(n_comp_1):
            mask_k = labels_1 == k
            if mask_k.sum() < GMM_N_COMPONENTS_2 * 3:
                continue
            fit_k = fit_pm_gmm(
                pm_phi[mask_k], pm_lam[mask_k],
                n_components=GMM_N_COMPONENTS_2,
                n_init=1, max_fit_samples=20_000, random_state=0,
            )
            if fit_k.gmm is None:
                continue
            gmm_label_2[mask_k] = fit_k.labels
            ok = fit_k.labels >= 0
            w = np.full(mask_k.sum(), np.nan)
            w[ok] = fit_k.widths[fit_k.labels[ok]]
            gmm_width_2[mask_k] = w

        sub["gmm_label_2"] = gmm_label_2
        sub["gmm_width_2"] = gmm_width_2
        subcell_dfs[cell_id] = sub
        print(f"===== CELL ID {cell_id} =====")
        display(sub.head())

    save_gmm_subcells(subcell_dfs, CKPT_DIR, subdir="cells_gmm2", width_thresh=None, verify=True)


In [ ]:
# Diagnostic only: do not pick a gmm_width_2 threshold on mock_frac.
# Stage 2 keeps every (cell, component) chunk. This sweep is leftover from the old cut.
MIN_MOCK_STARS = 0
components_to_use = []
for cell_id, cell_df in subcell_dfs.items():
    for (k, j), sub in cell_df.groupby(["gmm_label", "gmm_label_2"]):
        if k < 0 or j < 0:
            continue
        n_mock = sub["is_mock_stream"].to_numpy(bool).sum()
        if n_mock >= MIN_MOCK_STARS:
            components_to_use.append(sub)

print(len(components_to_use))

if not components_to_use:
    print("No second-level components with >= MIN_MOCK_STARS")
else:
    all_stars = pd.concat(components_to_use, ignore_index=True)
    gmm_width_2 = all_stars["gmm_width_2"].to_numpy()
    is_mock = all_stars["is_mock_stream"].to_numpy(bool)
    n_mock_total = int(is_mock.sum())
    n_other_total = int((~is_mock).sum())

    gmm_width_thresholds = np.linspace(0.1, 2.5, 25)
    rows = []
    for thresh in gmm_width_thresholds:
        keep = gmm_width_2 <= thresh
        n_mock = int((is_mock & keep).sum())
        n_other = int(((~is_mock) & keep).sum())
        rows.append({
            "threshold": thresh,
            "n_mock": n_mock,
            "n_other": n_other,
            "mock_frac": n_mock / n_mock_total if n_mock_total > 0 else np.nan,
            "other_frac": n_other / n_other_total if n_other_total > 0 else np.nan,
            "purity": n_mock / (n_mock + n_other) if (n_mock + n_other) > 0 else np.nan,
        })
    df_sweep = pd.DataFrame(rows)

    print("===== Points with mock_frac>0.9 and other_frac<1 =====")
    display(df_sweep[(df_sweep.mock_frac > 0.9) & (df_sweep.other_frac < 0.8)])

    chosen = df_sweep.iloc[12]

    fig, axs = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

    ax = axs[0]
    ax.plot(df_sweep.other_frac, df_sweep.mock_frac, "o-", markersize=4)
    ax.set_xlabel("Other kept")
    ax.set_ylabel("Mock kept")
    ax.set_title("GMM Width 2 Upper Bound")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    ax.axvline(chosen["other_frac"], label=f"Threshold={chosen['threshold']:.2f}", color="red", linestyle="--")
    ax.legend(loc="best")

    ax = axs[1]
    ax.plot(df_sweep.threshold, df_sweep.purity, "o-", markersize=4)
    ax.set_xlabel("GMM width 2 upper bound (mas/yr)")
    ax.set_ylabel("Purity")
    ax.set_title("Purity vs. GMM Width 2 Upper Bound")
    ax.grid(True, alpha=0.3)
    ax.axvline(chosen["threshold"], label=f"Mock Eff={chosen['mock_frac']:.2f}", color="red", linestyle="--")
    ax.legend(loc="best")

    plt.tight_layout()
    plt.show()


In [ ]:
selected_cells = {}
for name, cell in subcell_dfs.items():
    if ((cell.is_mock_stream.sum() / len(cell)) > 0.005) & (cell.gmm_width_2[0] < 1.3):
        selected_cells[name] = cell


In [ ]:
len(selected_cells)


In [ ]:
# NOTE: condensation MLP / graph_builder / cells index are NOT ported yet.
# This cell still references the upstream improved_interaction_networks package.
# Cutflow + plotting above use cpen.apps.streams.preprocess only.

# Train condensation MLP on gmm_2 cells (self-contained)
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from cpen.apps.streams.preprocess.config import TRAIN_VARS, TRAJ_VARS
from streams.data.cells import build_cell_index
from streams.data.graph_builder import build_knn_edges_by_cell_id
from streams.training.dataset import SupercellBatchDataset
from streams.models.condensation import CondenseModule

selected_cells = {}
for name, cell in subcell_dfs.items():
    if ((cell.is_mock_stream.sum() / len(cell)) > 0.005) and (cell.gmm_width_2.iloc[0] < 1.3):
        selected_cells[name] = cell

if len(selected_cells) == 0:
    raise ValueError("No selected_cells; run GMM stage 2 and ensure subcell_dfs has gmm_width_2.")

dfs = []
for i, (name, cell) in enumerate(selected_cells.items()):
    c = cell.copy()
    c["_cell_id"] = i
    dfs.append(c)
df = pd.concat(dfs, ignore_index=True)
cell_id = df["_cell_id"].to_numpy(np.int64)

for v in TRAIN_VARS:
    if v not in df.columns:
        if v == "ra_des_sin" and "ra_des" in df.columns:
            df["ra_des_sin"] = np.sin(np.deg2rad(df["ra_des"].to_numpy()))
        elif v == "ra_des_cos" and "ra_des" in df.columns:
            df["ra_des_cos"] = np.cos(np.deg2rad(df["ra_des"].to_numpy()))
        else:
            raise KeyError(f"Missing column {v} for training.")

idx = build_knn_edges_by_cell_id(df, cell_id_col="_cell_id", k_edges=16, k_pos=4, k_query=32, seed=0, verbose=False)

order, uniq_cells, cell_to_range = build_cell_index(cell_id)

def neighbor_indices_isolated(cell):
    rng = cell_to_range.get(cell)
    if rng is None:
        return np.empty(0, dtype=np.int64)
    s, e = rng
    return order[s:e]

X = df[TRAIN_VARS].to_numpy(dtype=np.float32)
X_t = torch.from_numpy(X)
train_mask = df["is_mock_stream"].to_numpy(bool)
stream_label_all = df["stream_label"].to_numpy(object)

dataset = SupercellBatchDataset(X_t=X_t, idx_global=idx, stream_label_all=stream_label_all, train_mask=train_mask, uniq_cells=uniq_cells, neighbor_indices_fn=neighbor_indices_isolated, seed=0)
loader = DataLoader(dataset, batch_size=None, num_workers=0, pin_memory=True)

ckpt_dir = Path(CKPT_DIR) / "checkpoints_gmm2"
ckpt_dir.mkdir(parents=True, exist_ok=True)

module = CondenseModule(in_dim=X.shape[1], width=512, depth=4, out_dim=64, eta_0=0.05, opt_name="adam", margin=1.0, neg_weight=1.0)

trainer = pl.Trainer(max_epochs=200, accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1, callbacks=[ModelCheckpoint(dirpath=str(ckpt_dir), filename="condense-gmm2-{epoch:03d}", save_top_k=1, monitor="train/loss_epoch", mode="min")], logger=CSVLogger(ckpt_dir, name="condense_gmm2"), log_every_n_steps=20)

print(f"Training on {len(selected_cells)} cells, {len(df):,} stars")
trainer.fit(module, train_dataloaders=loader)

